In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (LSTM, Dense)
from tensorflow.keras.layers import Input

In [2]:
df = pd.read_csv("../data/clean_traffic.csv")
df

,DateTime,Junction,Vehicles,ID,Hour,Month,Day,Weekday
0,2015-11-01 00:00:00,1,15,20151101001,0,11,1,6
1,2015-11-01 01:00:00,1,13,20151101011,1,11,1,6
2,2015-11-01 02:00:00,1,10,20151101021,2,11,1,6
3,2015-11-01 03:00:00,1,7,20151101031,3,11,1,6
4,2015-11-01 04:00:00,1,9,20151101041,4,11,1,6
...,...,...,...,...,...,...,...,...
48115,2017-06-30 19:00:00,4,11,20170630194,19,6,30,4
48116,2017-06-30 20:00:00,4,30,20170630204,20,6,30,4
48117,2017-06-30 21:00:00,4,16,20170630214,21,6,30,4
48118,2017-06-30 22:00:00,4,22,20170630224,22,6,30,4


In [3]:
df = df.sort_values(by=['Junction', 'DateTime'])

In [4]:
data = df['Vehicles'].values

In [5]:
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data.reshape(-1,1))

In [6]:
def create_sequence(data, sequence_length):
    X=[]
    y=[]
    for i in range(sequence_length, len(data)):
        X.append(data[i-sequence_length:i])
        y.append(data[i])
    return np.array(X), np.array(y)

In [7]:
sequence_length = 10
X, y = create_sequence(data_scaled, sequence_length)

In [8]:
print(X.shape)
print(y.shape)

(48110, 10, 1)
(48110, 1)


In [9]:
split_index = int(len(X)*0.8)
X_train, X_test, y_train, y_test = X[:split_index], X[split_index:], y[:split_index], y[split_index:]

In [10]:
model = Sequential([LSTM(50, activation = "relu", input_shape=(X_train.shape[1], 1)), Dense(1)])

d:\AI Smart City Intelligence System\ai_city_ml\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [11]:
model.compile(optimizer='adam', loss='mse')

In [12]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 50)             │        10,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,451 (40.82 KB)

 Trainable params: 10,451 (40.82 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))

Epoch 1/10
1203/1203 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0016 - val_loss: 9.8231e-04
Epoch 2/10
1203/1203 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 7.5648e-04 - val_loss: 9.1241e-04
Epoch 3/10
1203/1203 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - loss: 7.0053e-04 - val_loss: 9.1792e-04
Epoch 4/10
1203/1203 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - loss: 6.7936e-04 - val_loss: 8.8521e-04
Epoch 5/10
1203/1203 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - loss: 6.4811e-04 - val_loss: 8.5538e-04
Epoch 6/10
1203/1203 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 6.3084e-04 - val_loss: 8.7656e-04
Epoch 7/10
1203/1203 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 6.0941e-04 - val_loss: 8.2263e-04
Epoch 8/10
1203/1203 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 6.0216e-04 - val_loss: 8.2117e-04
Epoch 9/10
1203/1203 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 5.9480e-04 - val_loss: 8.2089e-04
Epoch 10/10
1203/1203 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 5.8950e-04 - val_loss: 8.1005e-04


In [14]:
predictions = model.predict(X_test)

301/301 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [15]:
predictions = scaler.inverse_transform(predictions)
y_test_actual = scaler.inverse_transform(y_test)

In [16]:
mae = mean_absolute_error(
    y_test_actual,
    predictions
)

mse = mean_squared_error(
    y_test_actual,
    predictions
)

r2 = r2_score(
    y_test_actual,
    predictions
)

print(f"MAE: {mae:.2f}")
print(f"MSE: {mse:.2f}")
print(f"R²: {r2:.2f}")

MAE: 2.91
MSE: 25.95
R²: 0.74
